In [1]:
import lsdb
import pandas as pd

[This PR](https://github.com/astronomy-commons/lsdb/pull/1581) is about a couple usability issues concerning ra/dec columns:

- "Guessing" ra/dec columns when someone creates a catalog from a dataframe
- More helpful error messages when ra/dec problems happen during crossmatch, or map_partitions

# Create catalog from dataframe

## (OLD) Literal "ra" or "dec" get matched

In [2]:
dummy_data = list(range(10))
df = pd.DataFrame({'ra': dummy_data, 'dec': dummy_data, 'col1': dummy_data})
cat = lsdb.from_dataframe(df)
print(f"ra_column: '{cat.hc_structure.catalog_info.ra_column}', "
      f"dec_column: '{cat.hc_structure.catalog_info.dec_column}'")
cat

ra_column: 'ra', dec_column: 'dec'


,ra,dec,col1
npartitions=1,,,
"Order: 1, Pixel: 19",int64[pyarrow],int64[pyarrow],int64[pyarrow]


## (OLD) 'ra' and 'dec' get matched case-insensitive, anywhere in columns

In [3]:
dummy_data = list(range(10))
df = pd.DataFrame({'col1': dummy_data, 'col2': dummy_data, 'Ra': dummy_data, 'DEC': dummy_data, 'col3': dummy_data})
cat = lsdb.from_dataframe(df)
print(f"ra_column: '{cat.hc_structure.catalog_info.ra_column}', "
      f"dec_column: '{cat.hc_structure.catalog_info.dec_column}'")
cat

ra_column: 'Ra', dec_column: 'DEC'


,col1,col2,Ra,DEC,col3
npartitions=1,,,,,
"Order: 1, Pixel: 19",int64[pyarrow],int64[pyarrow],int64[pyarrow],int64[pyarrow],int64[pyarrow]


## (NEW) known variations on ra/dec get matched
(Only in the first 4 columns)

In [4]:
dummy_data = list(range(10))
df = pd.DataFrame({ 'RAJ2000': dummy_data, 'decMean': dummy_data, 'col1': dummy_data, 'col2': dummy_data,'col3': dummy_data})
cat = lsdb.from_dataframe(df)
print(f"ra_column: '{cat.hc_structure.catalog_info.ra_column}', "
      f"dec_column: '{cat.hc_structure.catalog_info.dec_column}'")
cat

ra_column: 'RAJ2000', dec_column: 'decMean'


,RAJ2000,decMean,col1,col2,col3
npartitions=1,,,,,
"Order: 1, Pixel: 19",int64[pyarrow],int64[pyarrow],int64[pyarrow],int64[pyarrow],int64[pyarrow]


## (NEW) heuristic search for ra/dec matches
(Only in the first 4 columns)  
Note the warning!

In [5]:
dummy_data = list(range(10))
df = pd.DataFrame({ 'my.special.ra': dummy_data, 'my_special_dec': dummy_data, 'col1': dummy_data, 'col2': dummy_data,'col3': dummy_data})
cat = lsdb.from_dataframe(df)
print(f"ra_column: '{cat.hc_structure.catalog_info.ra_column}', "
      f"dec_column: '{cat.hc_structure.catalog_info.dec_column}'")
cat

ra_column: 'my.special.ra', dec_column: 'my_special_dec'


,my.special.ra,my_special_dec,col1,col2,col3
npartitions=1,,,,,
"Order: 1, Pixel: 19",int64[pyarrow],int64[pyarrow],int64[pyarrow],int64[pyarrow],int64[pyarrow]


## (OLD) If any ambiguous matches, error

In [6]:
dummy_data = list(range(10))
df = pd.DataFrame({ 'par.ra.lax': dummy_data, 'my_special_dec': dummy_data, 'col1': dummy_data, 'ra': dummy_data,'col3': dummy_data})
cat = lsdb.from_dataframe(df)
print(f"ra_column: '{cat.hc_structure.catalog_info.ra_column}', "
      f"dec_column: '{cat.hc_structure.catalog_info.dec_column}'")
cat

ValueError: Found 2 possible columns for 'ra': ['par.ra.lax', 'ra']. Please rename columns to disambiguate.

# Error message for bad crossmatch

In [ ]:
small_sky_catalog = lsdb.open_catalog('tests/data/small_sky')
small_sky_xmatch_catalog = lsdb.open_catalog('tests/data/small_sky_xmatch')

# Compute dataframes
left_dataframe = small_sky_catalog.compute()
right_dataframe = small_sky_xmatch_catalog.compute()   

# Rename ra and dec columns to abnormal names
right_dataframe_abnormal_ra_col = right_dataframe.rename(columns={"ra": "abnormal_rightasc_col_name"})
right_dataframe_abnormal_dec_col = right_dataframe.rename(columns={"dec": "abnormal_de_col_name"})

# Crossmatch method attempts to use default column names and fails
lsdb.crossmatch(
    left_dataframe,
    right_dataframe_abnormal_ra_col)

Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

Computing Catalog:   0%|          | 0/3 [00:00<?, ?it/s]

ValueError: No column found for 'ra' (required). You can supply ra/dec column names using the arguments `ra_column`, `dec_column`.

# Error messages for invalid ra/dec operations during map_partitions

# NOTE these checks are only implemented for `compute_single_partition == True`

In [ ]:
small_sky_source_catalog = lsdb.open_catalog('tests/data/small_sky_source')

In [ ]:
def rename_cols(df, names_in, names_out):
    """df = rename_cols(df, ['ra', 'dec'], ['my_ra', 'my_dec'])"""
    for name_in, name_out in zip(names_in, names_out):
        df[name_out] = df[name_in]
    col_names = [col for col in df.columns if col not in names_in]
    return df[col_names]


# Should raise because ra/dec column names change
small_sky_source_catalog.map_partitions(
    rename_cols, ["source_ra", "source_dec"], ["my_ra", "my_dec"], compute_single_partition=True
)

Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError: 'source_ra' not found in result. map_partitions() must not change names of ra or dec columns 'source_ra', 'source_dec'.

In [ ]:
def my_evil_function(df, col_name):
    df[col_name] = df[col_name] + 123
    return df


# Should raise because map_partitions() changes ra/dec values
small_sky_source_catalog.map_partitions(my_evil_function, 'source_ra', compute_single_partition=True)

Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError: ra/dec values have changed. map_partitions() must not change values of ra or dec columns 'source_ra', 'source_dec'.

In [ ]:
def my_evil_function(df, col_name):
    df[col_name] = df[col_name] + 123
    return df


# Should raise because map_partitions() changes ra/dec values
small_sky_source_catalog.map_partitions(my_evil_function, 'source_dec', compute_single_partition=True)

Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError: ra/dec values have changed. map_partitions() must not change values of ra or dec columns 'source_ra', 'source_dec'.

## Error message when map_partitions() returns a catalog with an invalid healpix index

In [ ]:
from hats.pixel_math.spatial_index import SPATIAL_INDEX_COLUMN

def my_evil_function(df, index_col):
        df[index_col] = df.index + 1
        return df.set_index(index_col)

# should raise because healpix index doesn't match ra/dec values
small_sky_source_catalog.map_partitions(
    my_evil_function, SPATIAL_INDEX_COLUMN, compute_single_partition=True
)

Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError: healpix index does not match ra/dec values. map_partitions() must not generate an invalid healpix index.